# Train YOLOv8n Drone V4 (Dataset Cận Cảnh / Close-Up Roboflow -> Inference Drone 3-4m)

**Phân tích bài toán Domain Mismatch:**
- **Dataset Roboflow:** Là các ảnh **cận cảnh (Close-Up)** chiếc lá / vết bệnh (lá chiếm 80-90% khung hình).
- **Thực tế Drone:** Chụp từ xa (3-4 mét), ảnh 6K/8K có hàng trăm chiếc lá nhỏ.

**Giải pháp tối ưu:**
1. **Train Model (`yolov8n.pt`):** Train trực tiếp trên dataset cận cảnh Roboflow (không cắt ảnh dataset kẻo rách lá), dùng `scale=0.8` để giả lập lá từ to đến nhỏ.
2. **Inference (SAHI / Tile):** Dùng `tile_and_detect.py` biến 1 ảnh drone lớn thành 200+ tile cận cảnh `640x640` / `1024x1024` + `ENHANCE=1` (Sharpen+CLAHE) để khớp hoàn hảo với phân bố dataset cận cảnh đã học!

**Cấu hình Kaggle Settings ⚙:**
- *Accelerator*: **GPU T4 x2** (hoặc P100)
- *Internet*: **ON**
- *Add secret* (ổ khóa bên phải): Tạo secret `ROBOFLOW_API_KEY`.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics roboflow sahi opencv-python

from ultralytics import YOLO
import ultralytics, torch
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 1. Tải Dataset Cận Cảnh từ Roboflow

In [ ]:
def _get_roboflow_key():
    import os
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        if key: return key
    except Exception: pass
    if os.environ.get("ROBOFLOW_API_KEY"): return os.environ["ROBOFLOW_API_KEY"]
    try:
        for line in open(".env"):
            if line.startswith("ROBOFLOW_API_KEY") and "=" in line:
                return line.split("=", 1)[1].strip()
    except FileNotFoundError: pass
    raise ValueError("Thiếu ROBOFLOW_API_KEY. Thêm secret tên ROBOFLOW_API_KEY trên Kaggle!")

ROBOFLOW_API_KEY = _get_roboflow_key()
print("Đã lấy ROBOFLOW_API_KEY")

from roboflow import Roboflow
WORKSPACE       = "trantungbach26-gmail-com"
PROJECT_NAME    = "citrus-disease-detection-yoydc-ahtka"
PROJECT_VERSION = 1

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT_NAME)
dataset_info = project.version(PROJECT_VERSION).download("yolov8")
print("Dataset cận cảnh đã tải về /kaggle/working/")

In [ ]:
import os, glob, yaml

candidates = glob.glob("/kaggle/working/*/data.yaml")
DATASET_PATH = os.path.dirname(candidates[0]) if candidates else "/kaggle/working/citrus-disease-detection-1"
TRAIN_DATA_YAML = os.path.join(DATASET_PATH, "data.yaml")
print("DATASET_PATH =", DATASET_PATH)

with open(TRAIN_DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print("Số class:", cfg["nc"])
print("Tên class:", cfg["names"])

## 2. Load Model & Train từ đầu (`yolov8n.pt` cho K230 Drone)

Model `yolov8n.pt` siêu nhẹ (3.2M params) để export ra `.kmodel` chạy mượt mà trên chip **Kendryte K230** của Drone.

In [ ]:
MODEL_NAME = "yolov8n.pt"
model = YOLO(MODEL_NAME)
print(f"Đã load pretrained backbone {MODEL_NAME} thành công!")

In [ ]:
# ===== CẤU HÌNH TRAIN CHO DATASET CẬN CẢNH ROBOFLOW =====
EPOCHS   = 150
IMGSZ    = 640        # Chuẩn resolution cho K230 ONNX export
BATCH    = 32         # Batch size 32 mượt mà với yolov8n
PATIENCE = 20

OUT_DIR = "/kaggle/working/drone_yolo_v4_close_up"
os.makedirs(OUT_DIR, exist_ok=True)

# Auto-backup best.pt sau mỗi epoch
import shutil
from ultralytics.utils import callbacks

def _backup(trainer):
    try:
        src = os.path.join(trainer.save_dir, "weights", "best.pt")
        shutil.copy(src, os.path.join(OUT_DIR, "best_checkpoint.pt"))
        print(f"  [backup epoch {trainer.epoch}] -> {OUT_DIR}/best_checkpoint.pt", flush=True)
    except Exception as e:
        pass

callbacks.default_callbacks["on_fit_epoch_end"].append(_backup)

print(f"Train {MODEL_NAME}: epochs={EPOCHS}, imgsz={IMGSZ}, batch={BATCH}, patience={PATIENCE}")

# Augmentations giúp model cận cảnh thích nghi với ảnh lá nhỏ từ xa
train_args = dict(
    data=TRAIN_DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    seed=42,
    time=8,              # Tối đa 8 giờ session Kaggle
    cache=True,
    workers=2,
    # Augmentations biến đổi scale & góc chụp
    scale=0.8,           # Thu nhỏ/phóng to ngẫu nhiên từ 20% đến 180% kích thước lá
    fliplr=0.5,          # Lật ngang
    mosaic=1.0,          # Mosaic 4 ảnh cận cảnh ghép lại (tạo góc nhìn nhiều lá)
    mixup=0.15,          # Trộn ảnh tạo nhiễu ánh sáng
    copy_paste=0.2,      # Trộn vết bệnh
    cos_lr=True,         # Cosine LR decay
    project="/kaggle/working/runs",
    name="drone_yolov8n_closeup",
)

results = model.train(**train_args)

## 3. Đánh giá & Export Model (pt + onnx)

File **`best.onnx`** sẽ được export để phục vụ chuyển đổi sang **`best.kmodel`** chạy trên chip Kendryte K230 của Drone.

In [ ]:
metrics = model.val()
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

In [ ]:
# Export ONNX dành cho chip K230 Drone
RESULTS_DIR = "/kaggle/working/runs/drone_yolov8n_closeup/weights"
best_path   = os.path.join(RESULTS_DIR, "best.pt")

if os.path.exists(best_path):
    best_model = YOLO(best_path)
    best_model.export(format="onnx", imgsz=IMGSZ, opset=11, simplify=True)
    shutil.copy(best_path, os.path.join(OUT_DIR, "best.pt"))
    onnx_src = os.path.join(RESULTS_DIR, "best.onnx")
    if os.path.exists(onnx_src):
        shutil.copy(onnx_src, os.path.join(OUT_DIR, "best.onnx"))
    print(f"Đã copy best.pt và best.onnx vào {OUT_DIR}")
else:
    print(f"Không tìm thấy {best_path}")

print("\n>>> TẢI KẾT QUẢ: Panel bên phải tab 'Output' -> biểu tượng Download all.")